In [2]:
import pandas as pd
import numpy as np

# 1. Carregar os dados (caminho ajustado para a pasta 'data')
caminho_pedidos = 'data\olist_orders_dataset.csv'
df_pedidos = pd.read_csv(caminho_pedidos)

# 2. Limpeza Inicial: Manter apenas os pedidos entregues
df_entregues = df_pedidos[df_pedidos['order_status'] == 'delivered'].copy()

# 3. Conversão de Tipos: Transformar o texto em data oficial (SLA)
colunas_de_data = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for coluna in colunas_de_data:
    df_entregues[coluna] = pd.to_datetime(df_entregues[coluna])

# 4. Calcular o atraso em dias (Regra de Negócio)
df_entregues['dias_variacao_sla'] = (df_entregues['order_delivered_customer_date'] - df_entregues['order_estimated_delivery_date']).dt.days

# 5. Criar status visual para facilitar os filtros no Power BI
df_entregues['status_entrega'] = np.where(df_entregues['dias_variacao_sla'] > 0, 'Atrasado', 'No Prazo')

# 6. Visualizar o resultado
colunas_visualizacao = ['order_id', 'order_estimated_delivery_date', 'order_delivered_customer_date', 'dias_variacao_sla', 'status_entrega']
display(df_entregues[colunas_visualizacao].head(10))

<>:5: SyntaxWarning: invalid escape sequence '\o'
<>:5: SyntaxWarning: invalid escape sequence '\o'
C:\Users\henri\AppData\Local\Temp\ipykernel_5048\2516533368.py:5: SyntaxWarning: invalid escape sequence '\o'
  caminho_pedidos = 'data\olist_orders_dataset.csv'


,order_id,order_estimated_delivery_date,order_delivered_customer_date,dias_variacao_sla,status_entrega
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-18,2017-10-10 21:25:13,-8.0,No Prazo
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-13,2018-08-07 15:27:45,-6.0,No Prazo
2,47770eb9100c2d0c44946d9cf07ec65d,2018-09-04,2018-08-17 18:06:29,-18.0,No Prazo
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-15,2017-12-02 00:28:42,-13.0,No Prazo
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-26,2018-02-16 18:17:02,-10.0,No Prazo
5,a4591c265e18cb1dcee52889e2d8acc3,2017-08-01,2017-07-26 10:57:55,-6.0,No Prazo
7,6514b8ad8028c9f2cc2374ded245783f,2017-06-07,2017-05-26 12:55:51,-12.0,No Prazo
8,76c6e866289321a7c93b82b54852dc33,2017-03-06,2017-02-02 14:08:10,-32.0,No Prazo
9,e69bfb5eb88e0ed6a785585b27e16dbf,2017-08-23,2017-08-16 17:14:30,-7.0,No Prazo
10,e6ce16cb79ec1d90b1da9085a6118aeb,2017-06-07,2017-05-29 11:18:31,-9.0,No Prazo


In [5]:
# 1. Calcular a porcentagem total de entregas No Prazo vs Atrasadas (OTD)
resumo_sla = df_entregues['status_entrega'].value_counts(normalize=True) * 100

print("--- Saúde da Operação (SLA Geral) ---")
print(resumo_sla.round(2).astype(str) + '%')

# 2. Descobrir a gravidade do problema: quando atrasa, atrasa por quanto tempo?
pedidos_atrasados = df_entregues[df_entregues['status_entrega'] == 'Atrasado']
media_dias_atraso = pedidos_atrasados['dias_variacao_sla'].mean()

print(f"\nAlerta Logístico: Quando uma entrega falha, ela chega em média {media_dias_atraso:.1f} dias atrasada.")

--- Saúde da Operação (SLA Geral) ---
status_entrega
No Prazo    93.23%
Atrasado     6.77%
Name: proportion, dtype: object

Alerta Logístico: Quando uma entrega falha, ela chega em média 10.6 dias atrasada.


In [6]:
# 1. Carregar a tabela de dimensão (Clientes)
caminho_clientes = 'data/olist_customers_dataset.csv'
df_clientes = pd.read_csv(caminho_clientes)

# 2. Unir as tabelas usando a chave em comum 'customer_id' (Equivalente ao INNER JOIN do SQL)
df_completo = pd.merge(df_entregues, df_clientes, on='customer_id', how='inner')

# 3. Isolar a operação: Filtrar apenas os pacotes que sofreram atraso
df_apenas_atrasados = df_completo[df_completo['status_entrega'] == 'Atrasado']

# 4. Agrupar os atrasos por Estado (UF) para descobrir os maiores gargalos geográficos
piores_estados = df_apenas_atrasados['customer_state'].value_counts().head(5)

print("--- TOP 5 Estados com Maior Volume de Atrasos ---")
print(piores_estados)

--- TOP 5 Estados com Maior Volume de Atrasos ---
customer_state
SP    1820
RJ    1495
MG     519
BA     396
RS     325
Name: count, dtype: int64


In [ ]:
# 1. Construindo a TABELA FATO (f_Movimentacao_Logistica)
# Selecionamos apenas as colunas que importam para o rastreio da carga
colunas_fato = [
    'order_id',
    'customer_id',
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'dias_variacao_sla',
    'status_entrega'
]
f_movimentacao_logistica = df_entregues[colunas_fato].copy()

# 2. Construindo a TABELA DIMENSÃO (d_Clientes)
# Removemos clientes duplicados para que o Power BI consiga fazer o relacionamento 1 para N (Um para Muitos)
colunas_clientes = ['customer_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
d_clientes = df_clientes[colunas_clientes].drop_duplicates()

# 3. Exportando os dados limpos para a pasta do projeto
# O Power BI vai ler estes ficheiros, que já não têm nulos ou lixo
f_movimentacao_logistica.to_csv('data/f_Movimentacao_Logistica.csv', index=False)
d_clientes.to_csv('data/d_Clientes.csv', index=False)

print("Tabelas Fato e Dimensão criadas e salvas com sucesso na pasta 'data'!")

Tabelas Fato e Dimensão criadas e salvas com sucesso na pasta 'data'!


In [ ]:
# 1. Lead Time do Armazém (Tempo entre a aprovação da compra e a entrega à transportadora)
df_entregues['lead_time_armazem'] = (df_entregues['order_delivered_carrier_date'] - df_entregues['order_approved_at']).dt.days

# 2. Lead Time da Transportadora (Tempo de trânsito puro - Transit Time)
df_entregues['transit_time_transportadora'] = (df_entregues['order_delivered_customer_date'] - df_entregues['order_delivered_carrier_date']).dt.days

# 3. Atualizar as colunas da nossa Tabela Fato para incluir as novas métricas
colunas_fato_expandida = [
    'order_id',
    'customer_id',
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'dias_variacao_sla',
    'status_entrega',
    'lead_time_armazem',
    'transit_time_transportadora'
]
f_movimentacao_logistica = df_entregues[colunas_fato_expandida].copy()

# 4. Exportar novamente o CSV atualizado para a pasta 'data'
f_movimentacao_logistica.to_csv('data/f_Movimentacao_Logistica.csv', index=False)

print("Métricas de Lead Time calculadas e Tabela Fato atualizada com sucesso!")

Métricas de Lead Time calculadas e Tabela Fato atualizada com sucesso!
